<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/LLMForBadagaTranslation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Run to install the desired libraries
!pip install peft==0.8.2
!pip install datasets==2.16.1

# We will use a simple 1b model which can fit in 15GB free TPU memory

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
import torch

model_name="bigscience/bloom-1b1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU
foundation_model.to("cuda")

# Let us see the output of the base model

In [3]:
def get_outputs(model, inputs, max_new_tokens=100):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.5,  # We don't want the repetition to happen
        early_stopping=True,  # Stop before reaching max_length
        eos_token_id=tokenizer.eos_token_id,
        num_beams=3
    )
    return outputs

# Inference original model
input_sentences = tokenizer("How are you?", return_tensors="pt")

# Move inputs to GPU
input_sentences = {k: v.to("cuda") for k, v in input_sentences.items()}

foundational_outputs_sentence = get_outputs(foundation_model, input_sentences, max_new_tokens=100)

print(tokenizer.batch_decode(foundational_outputs_sentence, skip_special_tokens=True))

['How are you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”\n\n“Good.”\n\n“And you?”']


In [4]:
# Custom dataset for translation
import pandas as pd
from datasets import Dataset

In [34]:
custom_prompts = [
    {
        "prompt": "I want you to act as a linux terminal### ਮੈਂ ਚਾਹੁੰਦਾ ਹਾਂ ਕਿ ਤੁਸੀਂ ਇੱਕ ਲਿਨਕਸ ਟਰਮੀਨਲ ਵਜੋਂ ਕੰਮ ਕਰੋ।"
    },
    {
        "prompt": "Explain the concept of machine learning### ਮਸ਼ੀਨ ਲਰਨਿੰਗ ਦੇ ਅਸੂਲ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Translate this sentence into Spanish### ਇਹ ਵਾਕ ਨੂੰ ਸਪੈਨਿਸ਼ ਵਿੱਚ ਅਨੁਵਾਦ ਕਰੋ।"
    },
    {
        "prompt": "Create a summary of this text### ਇਸ ਪਾਠ ਦਾ ਇਕ ਸੰਖੇਪ ਬਣਾਓ।"
    },
    {
        "prompt": "Generate a list of 5 healthy foods### 5 ਸਿਹਤਮੰਦ ਖਾਣੇ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the process of photosynthesis### ਫੋਟੋਸਿੰਥੈਸਿਸ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of regular exercise### ਨਿਯਮਿਤ ਵਰਜ਼ਿਸ਼ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Find the capital city of France### ਫ੍ਰਾਂਸ ਦੀ ਰਾਜਧਾਨੀ ਸ਼ਹਿਰ ਲੱਭੋ।"
    },
    {
        "prompt": "How do you solve a quadratic equation### ਤੁਸੀਂ ਇੱਕ ਚਤੁਰਭੁਜ ਸਮੀਕਰਨ ਨੂੰ ਕਿਵੇਂ ਹੱਲ ਕਰਦੇ ਹੋ###"
    },
    {
        "prompt": "List the top 10 programming languages in 2024### 2024 ਵਿੱਚ ਸਿਖਰ ਦੇ 10 ਪ੍ਰੋਗ੍ਰਾਮਿੰਗ ਭਾਸ਼ਾਵਾਂ ਦੀ ਸੂਚੀ ਦਿਓ।"
    },
    {
        "prompt": "Write a short story about a talking cat### ਇੱਕ ਬੋਲਦੇ ਬਿੱਲੀ ਬਾਰੇ ਇੱਕ ਛੋਟੀ ਕਹਾਣੀ ਲਿਖੋ।"
    },
    {
        "prompt": "Explain the theory of relativity in simple terms### ਰਿਲੇਟਿਵਿਟੀ ਦੇ ਸਿਧਾਂਤ ਨੂੰ ਸਧਾਰਨ ਸ਼ਬਦਾਂ ਵਿੱਚ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a poem about the moonlight### ਚੰਦਨੀ ਬਾਰੇ ਇੱਕ ਕਵਿਤਾ ਬਣਾਓ।"
    },
    {
        "prompt": "What are some good pickup lines### ਕੁਝ ਵਧੀਆ ਪਿਕਅੱਪ ਲਾਈਨਾਂ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the perfect vacation spot### ਸਬ ਤੋਂ ਵਧੀਆ ਛੁੱਟੀਆਂ ਵਾਲੀ ਥਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the top 10 ways to stay productive### ਉਤਪਾਦਕ ਬਣੇ ਰਹਿਣ ਦੇ ਸਿਖਰ ਦੇ 10 ਤਰੀਕੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Create a dialogue between a robot and a human### ਰੋਬੋਟ ਅਤੇ ਮਨੁੱਖ ਦਰਮਿਆਨ ਇੱਕ ਸੰਵਾਦ ਬਣਾਓ।"
    },
    {
        "prompt": "Summarize the plot of 'Hamlet'### 'ਹੈਮਲੇਟ' ਦੀ ਕਥਾ ਦਾ ਸੰਖੇਪ ਦਿਓ।"
    },
    {
        "prompt": "Write a motivational speech for students### ਵਿਦਿਆਰਥੀਆਂ ਲਈ ਇੱਕ ਪ੍ਰੇਰਣਾਤਮਕ ਭਾਸ਼ਣ ਲਿਖੋ।"
    },
    {
        "prompt": "Explain the basics of blockchain technology### ਬਲੌਕਚੇਨ ਤਕਨਾਲੋਜੀ ਦੇ ਮੂਲ ਤੱਤ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the lifecycle of a butterfly### ਇੱਕ ਤਿਤਲੀ ਦੀ ਜ਼ਿੰਦਗੀ ਚੱਕਰ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the health benefits of yoga### ਯੋਗ ਦੇ ਸਿਹਤ ਲਾਭ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the process of photosynthesis### ਫੋਟੋਸਿੰਥੈਸਿਸ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the function of the heart### ਦਿਲ ਦੀ ਕਾਰਜ ਸ਼ੀਲਤਾ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the main causes of climate change### ਮੌਸਮ ਪਰੀਵਰਤਨ ਦੇ ਮੁੱਖ ਕਾਰਨ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Create a list of the 7 wonders of the world### ਦੁਨੀਆ ਦੇ 7 ਅਜੂਬਿਆਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Explain the importance of sleep### ਨੀਂਦ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the symptoms of the common cold### ਆਮ ਜ਼ੁਕਾਮ ਦੇ ਲੱਛਣ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the process of water filtration### ਪਾਣੀ ਦੇ ਫਿਲਟ੍ਰੇਸ਼ਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the significance of the Great Wall of China### ਚੀਨ ਦੀ ਮਹਾਨ ਦਿਵਾਰ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a creative story about a haunted house### ਇੱਕ ਭੂਤ ਬੰਗਲੇ ਬਾਰੇ ਇੱਕ ਰਚਨਾਤਮਕ ਕਹਾਣੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the key features of a smartphone### ਇੱਕ ਸਮਾਰਟਫੋਨ ਦੀ ਮੁੱਖ ਵਿਸ਼ੇਸ਼ਤਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of a balanced diet### ਸੰਤੁਲਿਤ ਖੁਰਾਕ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the water cycle### ਪਾਣੀ ਦੇ ਚੱਕਰ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What is the significance of the Taj Mahal### ਤਾਜ ਮਹਲ ਦੀ ਮਹੱਤਤਾ ਕੀ ਹੈ###"
    },
    {
        "prompt": "Generate a list of 10 famous inventors### 10 ਮਸ਼ਹੂਰ ਖੋਜਕਾਰਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the process of making chocolate### ਚਾਕਲੇਟ ਬਣਾਉਣ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the importance of education### ਸਿੱਖਿਆ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the different types of renewable energy### ਨਵੀਨੀਕਰਣ ਯੋਗ ਊਰਜਾ ਦੇ ਵੱਖ ਵੱਖ ਤਰੀਕੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the anatomy of the human brain### ਮਨੁੱਖੀ ਦਿਮਾਗ ਦੀ ਬਣਾਵਟ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the top 5 tourist destinations in India### ਭਾਰਤ ਵਿੱਚ ਸਿਖਰ ਦੇ 5 ਸੈਲਾਨੀ ਮੰਜ਼ਿਲਾਂ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the process of cell division### ਸੈੱਲ ਵਿਭਾਜਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the role of the Prime Minister in a country### ਕਿਸੇ ਦੇਸ਼ ਵਿੱਚ ਪ੍ਰਧਾਨ ਮੰਤਰੀ ਦੀ ਭੂਮਿਕਾ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of meditation### ਧਿਆਨ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Generate a list of the 10 largest animals on Earth### ਧਰਤੀ ਦੇ 10 ਸਭ ਤੋਂ ਵੱਡੇ ਜਾਨਵਰਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the main events of World War II### ਦੂਜੀ ਵਿਸ਼ਵ ਯੁੱਧ ਦੇ ਮੁੱਖ ਘਟਨਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the concept of artificial intelligence### ਕ੍ਰਿਤ੍ਰਿਮ ਬੁੱਧਿਮੱਤਾ ਦੇ ਅਸੂਲ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the different types of ecosystems### ਵੱਖ ਵੱਖ ਤਰ੍ਹਾਂ ਦੇ ਪਰਿਸ਼ਥਿਤਿਕੀ ਤੰਦਰੁਸਤੀ ਸਿਸਟਮ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the cultural significance of Diwali### ਦਿਵਾਲੀ ਦੀ ਸੱਭਿਆਚਾਰਕ ਮਹੱਤਤਾ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the process of volcanic eruption### ਜਵਾਲਾਮੁਖੀ ਵਿਸਫੋਟ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the health benefits of drinking water### ਪਾਣੀ ਪੀਣ ਦੇ ਸਿਹਤ ਲਾਭ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the history of the Internet### ਇੰਟਰਨੈਟ ਦੀ ਇਤਿਹਾਸ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the principles of democracy### ਲੋਕਤੰਤਰ ਦੇ ਸਿਧਾਂਤ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the main components of a computer### ਕੰਪਿਊਟਰ ਦੇ ਮੁੱਖ ਹਿੱਸੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the process of making bread### ਬ੍ਰੇਡ ਬਣਾਉਣ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the significance of the Eiffel Tower### ਆਈਫਲ ਟਾਵਰ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a list of the top 10 books of all time### ਸਭ ਸਮੇਂ ਦੇ ਸਿਖਰ ਦੇ 10 ਕਿਤਾਬਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the stages of human development### ਮਨੁੱਖੀ ਵਿਕਾਸ ਦੇ ਪੜਾਅ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of learning a second language### ਦੂਜੀ ਭਾਸ਼ਾ ਸਿੱਖਣ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the process of recycling### ਰੀਸਾਈਕਲਿੰਗ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the features of the Amazon rainforest### ਅਮੇਜ਼ਨ ਵਰਖਾ ਜੰਗਲ ਦੀਆਂ ਵਿਸ਼ੇਸ਼ਤਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the top 5 benefits of exercise### ਕਸਰਤ ਦੇ ਸਿਖਰ ਦੇ 5 ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the theory of evolution### ਵਿਕਾਸ ਦੇ ਸਿਧਾਂਤ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the main functions of the liver### ਜਿਗਰ ਦੀਆਂ ਮੁੱਖ ਕਾਰਜਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the different types of pollution### ਪ੍ਰਦੂਸ਼ਣ ਦੇ ਵੱਖ ਵੱਖ ਤਰੀਕੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the significance of the Great Barrier Reef### ਮਹਾਨ ਬੈਰੀਅਰ ਰੀਫ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a list of the 10 smallest countries in the world### ਦੁਨੀਆ ਦੇ 10 ਸਭ ਤੋਂ ਛੋਟੇ ਦੇਸ਼ਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the process of fermentation### ਖਮੀਰਕਰਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of using renewable energy sources### ਨਵੀਨੀਕਰਣ ਯੋਗ ਊਰਜਾ ਸ੍ਰੋਤਾਂ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the importance of biodiversity### ਜੈਵ ਵਿਭਿੰਨਤਾ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    }
]

In [16]:
#custom_prompts = custom_prompts + custom_prompts2 + custom_prompts3 + custom_prompts4 + custom_prompts5 + custom_prompts6 + custom_prompts7 + custom_prompts8 + custom_prompts9 + custom_prompts10 + custom_prompts11

In [35]:
# Convert the custom prompts to a dataset
df = pd.DataFrame(custom_prompts)
dataset = Dataset.from_pandas(df)

#tokenized_dataset = dataset.map(lambda samples: tokenizer(samples["prompt"], add_special_tokens=True), batched=True)
tokenized_dataset = dataset.map(lambda samples: tokenizer(samples["prompt"], add_special_tokens=True), batched=True)
# Select a sample for training
train_sample = tokenized_dataset

# Display the tokenized samples
print(train_sample)

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'input_ids', 'attention_mask'],
    num_rows: 70
})


In [36]:
print(train_sample[0])

{'prompt': 'I want you to act as a linux terminal### ਮੈਂ ਚਾਹੁੰਦਾ ਹਾਂ ਕਿ ਤੁਸੀਂ ਇੱਕ ਲਿਨਕਸ ਟਰਮੀਨਲ ਵਜੋਂ ਕੰਮ ਕਰੋ।', 'input_ids': [44, 4026, 1152, 427, 1769, 661, 267, 104105, 28434, 105311, 14715, 124220, 23868, 4627, 62475, 8174, 2011, 8710, 45171, 47792, 57755, 1028, 31828, 20016, 57769, 527], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [39]:
from transformers import DataCollatorForSeq2Seq



In [40]:
import peft
from peft import LoraConfig, get_peft_model, PeftModel

lora_config = LoraConfig(
    r=4,  # As bigger the R bigger the parameters to train.
    lora_alpha=1,  # A scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules=["query_key_value"],  # You can obtain a list of target modules in the URL above.
    lora_dropout=0.05,  # Helps to avoid Overfitting.
    bias="lora_only",  # This specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

trainable params: 589,824 || all params: 1,065,904,128 || trainable%: 0.055335558283905996
None
